# Tabela global de resultados de teste

Este notebook monta tabelas com os resultados de teste do modelo `Llama3.1-I`, separadas por metodo de recomendacao, juntando:
- o resultado `without_optimization`, exibido na primeira linha de cada metodo;
- e todos os resultados `with_optimization` encontrados para as configuracoes de `out/prompt_optimization/Llama3.1-I`.

A ideia aqui e ter uma visao consolidada por algoritmo, cobrindo `bprmf`, `item_knn`, `ncf` e `user_knn`.

In [1]:
import json
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "run_prompt_optimizer.py").exists() and (candidate / "out").exists():
            return candidate
    raise FileNotFoundError(
        "Nao foi possivel localizar a raiz de explainability-with-LLMs. "
        "Execute o notebook a partir do projeto ou de um subdiretorio dele."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_NAME = "Llama3.1-I"
PROMPT_OPT_ROOT = PROJECT_ROOT / "out" / "prompt_optimization" / MODEL_NAME
TEST_ROOT = PROJECT_ROOT / "out" / "test_explainability"
WITHOUT_OPT_TEST_ROOT = TEST_ROOT / "without_optimization" / MODEL_NAME
WITH_OPT_TEST_ROOT = TEST_ROOT / "with_optimization" / MODEL_NAME

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PROMPT_OPT_ROOT: {PROMPT_OPT_ROOT}")
print(f"WITHOUT_OPT_TEST_ROOT: {WITHOUT_OPT_TEST_ROOT}")
print(f"WITH_OPT_TEST_ROOT: {WITH_OPT_TEST_ROOT}")

PROJECT_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/prompt-optim-expl-rec/explainability-with-LLMs
PROMPT_OPT_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/prompt-optim-expl-rec/explainability-with-LLMs/out/prompt_optimization/Llama3.1-I
WITHOUT_OPT_TEST_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/prompt-optim-expl-rec/explainability-with-LLMs/out/test_explainability/without_optimization/Llama3.1-I
WITH_OPT_TEST_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/prompt-optim-expl-rec/explainability-with-LLMs/out/test_explainability/with_optimization/Llama3.1-I


In [3]:
def lambda_to_float(lambda_name: str | None) -> float | None:
    if not lambda_name or not lambda_name.startswith("mmr_lambda_"):
        return None
    value = lambda_name.replace("mmr_lambda_", "")
    return float(value.replace("_", "."))


def find_named_parent(path: Path, prefix: str, default: str | None = None) -> str | None:
    for parent in path.parents:
        if parent.name.startswith(prefix):
            return parent.name
    return default


def discover_prompt_optimization_algorithms(prompt_opt_root: Path) -> list[str]:
    if not prompt_opt_root.exists():
        return []
    return sorted(path.name for path in prompt_opt_root.iterdir() if path.is_dir())


def load_test_metadata(metadata_path: Path) -> dict:
    return json.loads(metadata_path.read_text())


def discover_test_algorithms(test_root: Path) -> list[str]:
    if not test_root.exists():
        return []
    return sorted(path.name for path in test_root.iterdir() if path.is_dir())


def discover_without_optimization_results(test_root: Path, valid_algorithms: list[str]) -> pd.DataFrame:
    rows = []

    for algorithm in valid_algorithms:
        metadata_path = test_root / algorithm / "sep" / "responses_metadata.json"
        if not metadata_path.exists():
            continue

        payload = load_test_metadata(metadata_path)
        args = payload.get("args", {})
        rows.append(
            {
                "optimization_mode": "without_optimization",
                "algorithm": algorithm,
                "metric": payload.get("metric", args.get("metric", "metric")),
                "metric_name": payload.get("metric_name", "METRIC"),
                "metric_value": payload.get("metric_value"),
                "repr_model": pd.NA,
                "early_profile": pd.NA,
                "mmr_lambda": pd.NA,
                "lambda_value": pd.NA,
                "mmr_pool": pd.NA,
                "prompt_source": payload.get("prompt_source", "desconhecido"),
                "llm_method": args.get("llm_method", "desconhecido"),
                "n_users": payload.get("n_users"),
                "time_to_explain": payload.get("time_to_explain"),
                "best_prompt_path": payload.get("best_prompt_path"),
                "responses_metadata_path": metadata_path,
            }
        )

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)


def discover_with_optimization_results(test_root: Path, valid_algorithms: list[str]) -> pd.DataFrame:
    rows = []

    for algorithm in valid_algorithms:
        algorithm_dir = test_root / algorithm
        if not algorithm_dir.exists():
            continue

        for metadata_path in sorted(algorithm_dir.rglob("responses_metadata.json")):
            payload = load_test_metadata(metadata_path)
            args = payload.get("args", {})
            mmr_lambda = find_named_parent(metadata_path, "mmr_lambda_", None)

            rows.append(
                {
                    "optimization_mode": "with_optimization",
                    "algorithm": algorithm,
                    "metric": payload.get("metric", args.get("metric", "metric")),
                    "metric_name": payload.get("metric_name", "METRIC"),
                    "metric_value": payload.get("metric_value"),
                    "repr_model": find_named_parent(metadata_path, "repr_", pd.NA),
                    "early_profile": find_named_parent(metadata_path, "early_", pd.NA),
                    "mmr_lambda": mmr_lambda or pd.NA,
                    "lambda_value": lambda_to_float(mmr_lambda),
                    "mmr_pool": find_named_parent(metadata_path, "mmr_pool_", pd.NA),
                    "prompt_source": payload.get("prompt_source", "desconhecido"),
                    "llm_method": args.get("llm_method", payload.get("best_prompt_model", "desconhecido")),
                    "n_users": payload.get("n_users"),
                    "time_to_explain": payload.get("time_to_explain"),
                    "best_prompt_path": payload.get("best_prompt_path"),
                    "responses_metadata_path": metadata_path,
                }
            )

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)


def build_consolidated_table(prompt_opt_root: Path, without_opt_test_root: Path, with_opt_test_root: Path) -> pd.DataFrame:
    algorithms = sorted(
        set(discover_prompt_optimization_algorithms(prompt_opt_root))
        | set(discover_test_algorithms(without_opt_test_root))
        | set(discover_test_algorithms(with_opt_test_root))
    )
    without_opt = discover_without_optimization_results(without_opt_test_root, algorithms)
    with_opt = discover_with_optimization_results(with_opt_test_root, algorithms)

    frames = [frame for frame in (without_opt, with_opt) if not frame.empty]
    if not frames:
        return pd.DataFrame()

    consolidated = pd.concat(frames, ignore_index=True)
    consolidated["optimization_order"] = consolidated["optimization_mode"].map(
        {"without_optimization": 0, "with_optimization": 1}
    )
    consolidated = consolidated.sort_values(
        by=["algorithm", "optimization_order", "repr_model", "lambda_value", "mmr_pool"],
        na_position="last",
    ).reset_index(drop=True)

    preferred_columns = [
        "optimization_mode",
        "algorithm",
        "llm_method",
        "metric",
        "metric_name",
        "metric_value",
        "repr_model",
        "early_profile",
        "mmr_lambda",
        "lambda_value",
        "mmr_pool",
        "prompt_source",
        "n_users",
        "time_to_explain",
        "best_prompt_path",
        "responses_metadata_path",
    ]
    return consolidated[preferred_columns]


In [4]:
consolidated_table = build_consolidated_table(
    prompt_opt_root=PROMPT_OPT_ROOT,
    without_opt_test_root=WITHOUT_OPT_TEST_ROOT,
    with_opt_test_root=WITH_OPT_TEST_ROOT,
)

if consolidated_table.empty:
    warnings.warn("Nenhum resultado de teste foi encontrado para montar a tabela global.")
else:
    print(f"Linhas na tabela consolidada: {len(consolidated_table)}")
    for algorithm, algorithm_table in consolidated_table.groupby("algorithm", sort=False):
        print(f"\nAlgoritmo: {algorithm}")
        display(algorithm_table.reset_index(drop=True))

Linhas na tabela consolidada: 28

Algoritmo: bprmf


/tmp/ipykernel_1084826/1581137898.py:121: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  consolidated = pd.concat(frames, ignore_index=True)


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,bprmf,Llama3.1-I,sep,SEP,0.642861,NaN,NaN,NaN,NaN,<NA>,default,122,220.432820,None,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
1,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.709069,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,237.684772,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
2,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.724264,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,242.912650,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
3,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.709069,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,225.470375,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
4,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.713813,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,209.179955,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
5,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.709069,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,226.330889,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
6,with_optimization,bprmf,Llama3.1-I,sep,SEP,0.709069,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,225.448173,out/prompt_optimization/Llama3.1-I/bprmf/sep/r...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...



Algoritmo: item_knn


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,item_knn,Llama3.1-I,sep,SEP,0.582436,NaN,NaN,NaN,NaN,<NA>,default,122,219.726482,None,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
1,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.708952,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,246.160230,out/prompt_optimization/Llama3.1-I/item_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
2,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.688671,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,264.427568,out/prompt_optimization/Llama3.1-I/item_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
3,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.688671,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,271.998394,out/prompt_optimization/Llama3.1-I/item_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
4,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.688671,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,264.786311,out/prompt_optimization/Llama3.1-I/item_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
5,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.708952,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,245.331824,out/prompt_optimization/Llama3.1-I/item_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
6,with_optimization,item_knn,Llama3.1-I,sep,SEP,0.688671,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,261.905214,out/prompt_optimization/Llama3.1-I/item_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...



Algoritmo: ncf


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,ncf,Llama3.1-I,sep,SEP,0.596754,NaN,NaN,NaN,NaN,<NA>,default,122,218.915147,None,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
1,with_optimization,ncf,Llama3.1-I,sep,SEP,0.635935,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,445.818260,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
2,with_optimization,ncf,Llama3.1-I,sep,SEP,0.642396,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,456.838761,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
3,with_optimization,ncf,Llama3.1-I,sep,SEP,0.693150,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,222.102134,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
4,with_optimization,ncf,Llama3.1-I,sep,SEP,0.668023,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,218.341605,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
5,with_optimization,ncf,Llama3.1-I,sep,SEP,0.644048,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,476.666018,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
6,with_optimization,ncf,Llama3.1-I,sep,SEP,0.690858,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,238.921718,out/prompt_optimization/Llama3.1-I/ncf/sep/rep...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...



Algoritmo: user_knn


,optimization_mode,algorithm,llm_method,metric,metric_name,metric_value,repr_model,early_profile,mmr_lambda,lambda_value,mmr_pool,prompt_source,n_users,time_to_explain,best_prompt_path,responses_metadata_path
0,without_optimization,user_knn,Llama3.1-I,sep,SEP,0.625719,NaN,NaN,NaN,NaN,<NA>,default,122,218.638526,None,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
1,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.716674,repr_llm2vec,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,218.103115,out/prompt_optimization/Llama3.1-I/user_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
2,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.716674,repr_llm2vec,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,218.640134,out/prompt_optimization/Llama3.1-I/user_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
3,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.697653,repr_llm2vec,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,245.840716,out/prompt_optimization/Llama3.1-I/user_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
4,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.716674,repr_sbert,early_false,mmr_lambda_0_0,0.0,mmr_pool_10,best_prompt,122,218.944408,out/prompt_optimization/Llama3.1-I/user_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
5,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.716674,repr_sbert,early_false,mmr_lambda_0_5,0.5,mmr_pool_10,best_prompt,122,217.073365,out/prompt_optimization/Llama3.1-I/user_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
6,with_optimization,user_knn,Llama3.1-I,sep,SEP,0.712782,repr_sbert,early_false,mmr_lambda_1_0,1.0,mmr_pool_10,best_prompt,122,253.251692,out/prompt_optimization/Llama3.1-I/user_knn/se...,/mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_W...
